In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image  




In [2]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),    
    transforms.RandomHorizontalFlip(),    
    transforms.ColorJitter(0.2,0.2,0.2),  
    transforms.ToTensor(),                
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])  
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])


In [3]:
train_dataset = datasets.ImageFolder(root='Training_set/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder(root='Training_set/val',   transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True)


In [4]:
model = models.resnet18(pretrained=True)
num_ftrs = model.fc.in_features
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(num_ftrs, num_classes)


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
criterion = nn.CrossEntropyLoss()           
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()                       
    running_loss = 0.0
    running_corrects = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)      
        labels = labels.to(device)

        optimizer.zero_grad()           
        outputs = model(inputs)         
        loss = criterion(outputs, labels) 
        loss.backward()                 
        optimizer.step()                

        # Metrics
        _, preds = torch.max(outputs, 1)  
        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)
        total += inputs.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = running_corrects.double() / total

    # Validation
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_loss = val_loss / len(val_dataset)
    val_acc  = val_corrects.double() / len(val_dataset)
    scheduler.step()   
    print(f"Epoch {epoch+1}: train_loss={epoch_loss:.4f}, train_acc={epoch_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 1: train_loss=0.1586, train_acc=0.9293, val_loss=0.0238, val_acc=0.9895
Epoch 2: train_loss=0.0996, train_acc=0.9630, val_loss=0.0516, val_acc=0.9843
Epoch 3: train_loss=0.0724, train_acc=0.9744, val_loss=0.0326, val_acc=0.9895
Epoch 4: train_loss=0.0671, train_acc=0.9697, val_loss=0.0278, val_acc=0.9869
Epoch 5: train_loss=0.0613, train_acc=0.9731, val_loss=0.0301, val_acc=0.9869
Epoch 6: train_loss=0.0474, train_acc=0.9805, val_loss=0.0220, val_acc=0.9948
Epoch 7: train_loss=0.0504, train_acc=0.9805, val_loss=0.0210, val_acc=0.9895
Epoch 8: train_loss=0.0416, train_acc=0.9825, val_loss=0.0199, val_acc=0.9921
Epoch 9: train_loss=0.0430, train_acc=0.9852, val_loss=0.0204, val_acc=0.9895
Epoch 10: train_loss=0.0331, train_acc=0.9879, val_loss=0.0205, val_acc=0.9921


In [7]:
from PIL import Image
img = Image.open("some_image.jpg").convert("RGB")
input_tensor = val_transforms(img).unsqueeze(0).to(device)  

model.eval()
with torch.no_grad():
    outputs = model(input_tensor)
    probs = torch.softmax(outputs, dim=1)
    top_prob, top_class = torch.max(probs, 1)
    predicted_label = train_dataset.classes[top_class.item()]
    print(predicted_label, top_prob.item())

electric car 0.9999794960021973


In [8]:
torch.save(model.state_dict(), 'model_weights.pth')       
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load('model_weights.pth'))
model.eval()


c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  